# MC dropout uncertainty
Uses the prepared thesis files, fixed-fusion checkpoints, dataset contract and architecture without changing the classifier. It first reproduces the existing deterministic out-of-fold outputs, then runs MC dropout without reactivating training-time modality dropout.


In [ ]:
# 1. Exact project paths, imports, and run controls

from pathlib import Path
from collections import defaultdict
import gc
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

# Exact project root used throughout the thesis experiments.
PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"

# Exact prepared data/schema locations.
FINAL_MODEL_SCHEMA_PATH = (
    MODEL_ROOT / "manifests" / "final_model_input_schema.json"
)
INPUT_ROOT = (
    MODEL_ROOT / "final_task_ready_inputs" / "mci_prognosis"
)

# Exact completed fixed-equal-fusion experiment used for the detailed
# uncertainty/conflict analysis in the dissertation.
FIXED_EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / "gated_cmt_fixed_equal_fusion_md050"
    / "mci_prognosis"
)

# Exact saved pooled OOF predictions from that experiment. We use these as a
# fail-fast validation that this notebook has reconstructed the same model.
OFFICIAL_POOLED_PREDICTIONS_PATH = (
    MODEL_ROOT
    / "experiments"
    / "gated_cmt_fixed_equal_fusion_md050"
    / "cross_validation_aggregation"
    / "mci_prognosis"
    / "pooled_predictions"
    / "pooled_out_of_fold_predictions_raw.csv"
)

# New outputs only. Existing thesis results are never overwritten.
OUTPUT_ROOT = (
    MODEL_ROOT
    / "experiment_comparisons"
    / "mc_dropout_vs_evidential_fixed_md050"
)
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Run controls
# Quick test: fold 0 with 10 MC samples.
# confirm the pipeline. Then change only QUICK_TEST=False and rerun all cells.
QUICK_TEST = True

OUTER_FOLDS = (0,) if QUICK_TEST else (0, 1, 2, 3, 4)
MC_SAMPLES = 10 if QUICK_TEST else 30
BATCH_SIZE = 2
NUM_WORKERS = 0
CLASSIFICATION_THRESHOLD = 0.50
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 20260928

# Optional branch-level MC attribution is expensive, so leave it off until the
# main 5-fold MC-dropout comparison is complete.
RUN_BRANCH_ATTRIBUTION = False
ATTRIBUTION_MC_SAMPLES = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Quick test:", QUICK_TEST)
print("Folds:", OUTER_FOLDS)
print("MC samples:", MC_SAMPLES)

# Fail immediately if the known files are not where this project stores them.
required_paths = [
    FINAL_MODEL_SCHEMA_PATH,
    OFFICIAL_POOLED_PREDICTIONS_PATH,
]
for fold in OUTER_FOLDS:
    required_paths.extend([
        INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv",
        FIXED_EXPERIMENT_ROOT / f"fold_{fold}" / "checkpoints" / "best_validation_auc_checkpoint.pt",
    ])

missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "The expected thesis files are missing from their known project locations:\n"
        + "\n".join(missing)
    )

with open(FINAL_MODEL_SCHEMA_PATH, "r", encoding="utf-8") as f:
    final_model_schema = json.load(f)

identifier_columns = final_model_schema["identifier_columns"]
target_column = final_model_schema["model_target_column"]
branch_mask_columns = final_model_schema["branch_mask_columns"]
feature_mask_columns = final_model_schema["feature_mask_columns"]

print("\nSchema:", FINAL_MODEL_SCHEMA_PATH)
print("Official pooled predictions:", OFFICIAL_POOLED_PREDICTIONS_PATH)
print("Fixed experiment:", FIXED_EXPERIMENT_ROOT)


In [ ]:
# 2. Exact data contract and architecture constants

MRI_SPATIAL_SHAPE = (177, 213, 183)
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)

MODALITY_EMBEDDING_DIM = 64
NUMBER_OF_CLASSES = 2

MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]

PROGNOSIS_CLASS_ORDER = ["sMCI", "pMCI"]

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


In [ ]:
# 3. Exact fixed-fusion dataset/model definitions
# These definitions are copied from the project's fixed-equal-fusion training
# notebook. No renamed constructor arguments or invented file contracts are used.

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # Reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # Retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # Preserve the mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri

    def __len__(self):
        return len(self.dataframe)

    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)

    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)

    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)

    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # Load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # Ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # Demographics

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # Cognitive and functional measures

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # CSF

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # Plasma

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # APOE

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # MRI

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }

        # Complete sample

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        values,
        feature_mask,
    ):
        # Pass both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original 3MT image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # Initial convolutional stem

        # Use two initial 3D convolutions, following the broad
        # structure shown in the 3MT image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original 3MT inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )

        # Four residual downsampling blocks

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )

        # Determine the resulting patch grid

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )

        # Learned positional embeddings

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        # Patch-wise transformer encoder

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )

        # Projection to the shared modality dimension

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # Flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # Add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )

    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )

    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # Add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask

    def forward(self, modalities):
        # Get the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # Mask every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # Return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )

    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
        branch_mask,
    ):
        # Use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "belief": effective_evidence / strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )

    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )

    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }

    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]

        # Conflict mass

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)

        # Fused belief and uncertainty masses

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation

        # Recover the fused Dirichlet opinion

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }

    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # Retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }

        # Final fused opinion

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(self, representation):
        return self.network(
            representation
        )

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )

    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # Intermediate auxiliary predictions

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )

        # Final joint evidential opinion

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final 3MT query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )

class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )

    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )

    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )

        # Modality-specific encoders

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )

        # Independent modality evidential pathway

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )

        # Interaction-aware 3MT pathway

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        # Final fixed 50/50 hybrid fusion

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )

    # Training-time modality dropout

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )

        # Sample branch-retention indicators

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )

        # Prevent complete information removal

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # Restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0

        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )

    # Construct effective modality dictionaries

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities

    # Complete forward pass

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # Apply training-time modality dropout

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )

        # Encode all six modalities

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. Use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]

        # Independent modality opinions and TMC fusion

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )

        # Interaction-aware 3MT pathway

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]

        # Final fixed-equal hybrid opinion

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )

        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


In [ ]:
# 4. Fold loading, checkpoint loading, and baseline extraction

def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def move_nested_batch_to_device(batch, device):
    if torch.is_tensor(batch):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, dict):
        return {k: move_nested_batch_to_device(v, device) for k, v in batch.items()}
    if isinstance(batch, list):
        return [move_nested_batch_to_device(v, device) for v in batch]
    if isinstance(batch, tuple):
        return tuple(move_nested_batch_to_device(v, device) for v in batch)
    return batch

def input_path(fold):
    return INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"

def checkpoint_path(fold):
    return (
        FIXED_EXPERIMENT_ROOT
        / f"fold_{fold}"
        / "checkpoints"
        / "best_validation_auc_checkpoint.pt"
    )

def make_test_loader(fold):
    table = pd.read_csv(input_path(fold))
    test_table = (
        table.loc[table["DATA_ROLE"] == "test"]
        .reset_index(drop=True)
    )

    expected = 109 if fold in (0, 1, 2, 3) else 108
    if len(test_table) != expected:
        raise ValueError(
            f"Fold {fold}: expected {expected} test participants, found {len(test_table)}."
        )

    dataset = ADNIMultimodalDataset(
        dataframe=test_table,
        column_contract=dataset_column_contract,
        branch_mask_order=branch_mask_columns,
        feature_mask_order=feature_mask_columns,
        target_column=target_column,
        load_mri=True,
    )

    loader = DataLoader(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    return test_table, loader

def load_fixed_model(fold):
    model = ADNIEvidential3MTTMCModel(
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ).to(DEVICE)

    path = checkpoint_path(fold)
    try:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=DEVICE)

    if "model_state_dict" not in checkpoint:
        raise KeyError(f"Fold {fold} checkpoint has no model_state_dict: {path}")

    # Strict loading is intentional. A mismatch means this is not the exact
    # architecture/checkpoint pair and the notebook must stop.
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    return model, checkpoint

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]

    # Exact maximum pairwise conflict across all genuinely available
    # modality opinions. This matches the thesis attribution analysis
    # and adds no new training or model parameters.
    pairwise_conflicts = []

    for first_index, first_modality in enumerate(MODALITY_ORDER):
        for second_index in range(first_index + 1, len(MODALITY_ORDER)):
            second_modality = MODALITY_ORDER[second_index]

            first_belief = model_output["modality_opinions"][
                first_modality
            ]["belief"]
            second_belief = model_output["modality_opinions"][
                second_modality
            ]["belief"]

            belief_outer = (
                first_belief.unsqueeze(-1)
                * second_belief.unsqueeze(-2)
            )
            total_product = belief_outer.sum(dim=(-2, -1))
            same_class = torch.diagonal(
                belief_outer,
                dim1=-2,
                dim2=-1,
            ).sum(dim=-1)

            pair_is_available = (
                model_output["effective_branch_masks"][:, first_index]
                * model_output["effective_branch_masks"][:, second_index]
            )

            pairwise_conflicts.append(
                (total_product - same_class) * pair_is_available
            )

    maximum_pairwise_conflict = torch.stack(
        pairwise_conflicts,
        dim=1,
    ).max(dim=1).values

    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "max_pairwise_conflict":
            maximum_pairwise_conflict
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "mean_tmc_conflict":
            final_output[
                "mean_tmc_conflict"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted

def deterministic_batch_table(batch, model_output, fold):
    exact = extract_batch_prediction_arrays(batch, model_output)
    table = pd.DataFrame({
        "FOLD": int(fold),
        "RID": exact["rid"].astype(np.int64),
        "TARGET": exact["target"].astype(np.int64),
        "FINAL_P_pMCI": exact["final_p_pMCI"],
        "FINAL_UNCERTAINTY": exact["final_uncertainty"],
        "MAX_PAIRWISE_CONFLICT": exact["max_pairwise_conflict"],
        "MEAN_TMC_CONFLICT": exact["mean_tmc_conflict"],
        "ORIGINAL_MODALITY_COUNT": exact["original_modality_count"].astype(np.int64),
    })
    table["FINAL_PREDICTED_CLASS"] = (
        table["FINAL_P_pMCI"] >= CLASSIFICATION_THRESHOLD
    ).astype(np.int64)
    table["ERROR"] = (
        table["FINAL_PREDICTED_CLASS"] != table["TARGET"]
    ).astype(np.int64)
    table["CONFLICT_AWARE_UNCERTAINTY"] = (
        1.0
        - (1.0 - table["FINAL_UNCERTAINTY"])
        * (1.0 - table["MAX_PAIRWISE_CONFLICT"])
    )
    return table


In [ ]:
# 5. MC-dropout activation that keeps modality dropout OFF

def set_deterministic_eval(model):
    model.eval()
    if model.training:
        raise RuntimeError("Model should be in evaluation mode.")

def enable_only_neural_dropout(model):
    """
    Keep the top-level model in eval mode so _apply_modality_dropout() returns
    the genuine branch masks, then activate only ordinary dropout-bearing
    modules used inside the neural network.
    """
    model.eval()

    stochastic_types = (
        nn.Dropout,
        nn.Dropout1d,
        nn.Dropout2d,
        nn.Dropout3d,
        nn.MultiheadAttention,
        nn.TransformerEncoder,
        nn.TransformerEncoderLayer,
    )

    activated = []
    for name, module in model.named_modules():
        if isinstance(module, stochastic_types):
            module.train()
            activated.append(name)

    # Crucial project-specific safety check: modality dropout checks the
    # top-level model.training flag. This must remain False.
    if model.training:
        raise RuntimeError(
            "Top-level model entered training mode; that would reactivate modality dropout."
        )
    return activated

def entropy(probabilities, eps=1e-12):
    p = np.clip(np.asarray(probabilities, dtype=np.float64), eps, 1.0)
    return -np.sum(p * np.log(p), axis=-1)

def run_stochastic_pass(model, batch, seed):
    # Explicit seeds make the MC experiment reproducible.
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    with torch.inference_mode():
        out = model(
            modalities=batch["modalities"],
            original_branch_masks=batch["branch_masks"].float(),
        )

    return out["final_output"]["probabilities"].detach().float().cpu().numpy()


## Main inference

Run this after the setup cells. In quick-test mode it evaluates only fold 0. If the official-output validation passes, set `QUICK_TEST = False` in Cell 1 and rerun from the top for all five folds.


In [ ]:
import math


In [ ]:
# 6. Deterministic reconstruction + MC-dropout inference

set_global_seed(20260928)

official_oof = pd.read_csv(OFFICIAL_POOLED_PREDICTIONS_PATH)
required_official_columns = {"FOLD", "RID", "TARGET", "FINAL_P_pMCI", "FINAL_UNCERTAINTY"}
missing_cols = required_official_columns - set(official_oof.columns)
if missing_cols:
    raise ValueError(
        f"Official pooled prediction table is missing columns: {sorted(missing_cols)}"
    )

all_rows = []
validation_rows = []

for fold in OUTER_FOLDS:
    print("\n" + "=" * 72)
    print(f"FOLD {fold}")
    print("=" * 72)

    _, loader = make_test_loader(fold)
    model, checkpoint = load_fixed_model(fold)
    checkpoint_epoch = int(checkpoint.get("epoch", -1))
    print("Checkpoint:", checkpoint_path(fold))
    print("Checkpoint epoch:", checkpoint_epoch)

    fold_deterministic = []
    fold_mc = []

    for batch_index, batch_cpu in enumerate(loader):
        batch = move_nested_batch_to_device(batch_cpu, DEVICE)

        # Exact deterministic pass first.
        set_deterministic_eval(model)
        with torch.inference_mode():
            det_output = model(
                modalities=batch["modalities"],
                original_branch_masks=batch["branch_masks"].float(),
            )

        # Prove modality dropout is off in deterministic evaluation.
        mask_diff = (
            det_output["effective_branch_masks"]
            - det_output["original_branch_masks"]
        ).abs().max().item()
        if mask_diff != 0.0:
            raise RuntimeError(
                f"Fold {fold}, batch {batch_index}: modality masks changed in eval mode."
            )

        det_table = deterministic_batch_table(batch, det_output, fold)
        fold_deterministic.append(det_table)

        # Activate only ordinary neural dropout. Top-level model stays eval.
        activated = enable_only_neural_dropout(model)
        if batch_index == 0:
            print(f"Activated {len(activated)} neural stochastic modules.")
            print("Top-level model.training:", model.training)
            print("Modality dropout probability stored in model:", model.modality_dropout_probability)

        sample_probabilities = []
        for mc_index in range(MC_SAMPLES):
            seed = 2026092800 + fold * 100000 + batch_index * 1000 + mc_index
            sample_probabilities.append(run_stochastic_pass(model, batch, seed))

        probs = np.stack(sample_probabilities, axis=0)  # [T, B, 2]
        mean_prob = probs.mean(axis=0)
        predictive_entropy = entropy(mean_prob)
        expected_entropy = entropy(probs).mean(axis=0)
        mutual_information = np.maximum(predictive_entropy - expected_entropy, 0.0)
        p_pmci_std = probs[:, :, 1].std(axis=0, ddof=1 if MC_SAMPLES > 1 else 0)

        rids = batch["rid"].detach().cpu().numpy().reshape(-1)
        for i, rid in enumerate(rids):
            fold_mc.append({
                "FOLD": int(fold),
                "RID": int(rid),
                "MC_MEAN_P_pMCI": float(mean_prob[i, 1]),
                "MC_P_pMCI_STD": float(p_pmci_std[i]),
                "MC_PREDICTIVE_ENTROPY": float(predictive_entropy[i]),
                "MC_EXPECTED_ENTROPY": float(expected_entropy[i]),
                "MC_MUTUAL_INFORMATION": float(mutual_information[i]),
            })

        del det_output, probs, mean_prob, sample_probabilities
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    det_fold = pd.concat(fold_deterministic, ignore_index=True)
    mc_fold = pd.DataFrame(fold_mc)
    merged = det_fold.merge(mc_fold, on=["FOLD", "RID"], validate="one_to_one")

    # Fail-fast validation against the already saved official OOF result.
    official_fold = official_oof.loc[official_oof["FOLD"] == fold, [
        "FOLD", "RID", "TARGET", "FINAL_P_pMCI", "FINAL_UNCERTAINTY"
    ]].copy()

    check = merged.merge(
        official_fold,
        on=["FOLD", "RID"],
        how="inner",
        suffixes=("_REBUILT", "_OFFICIAL"),
        validate="one_to_one",
    )

    if len(check) != len(merged):
        raise RuntimeError(
            f"Fold {fold}: rebuilt participants do not match official pooled OOF participants."
        )

    if not np.array_equal(
        check["TARGET_REBUILT"].to_numpy(),
        check["TARGET_OFFICIAL"].to_numpy(),
    ):
        raise RuntimeError(f"Fold {fold}: target mismatch with official OOF predictions.")

    max_prob_diff = float(np.max(np.abs(
        check["FINAL_P_pMCI_REBUILT"] - check["FINAL_P_pMCI_OFFICIAL"]
    )))
    max_u_diff = float(np.max(np.abs(
        check["FINAL_UNCERTAINTY_REBUILT"] - check["FINAL_UNCERTAINTY_OFFICIAL"]
    )))

    validation_rows.append({
        "FOLD": fold,
        "N": len(check),
        "MAX_ABS_PROBABILITY_DIFFERENCE": max_prob_diff,
        "MAX_ABS_UNCERTAINTY_DIFFERENCE": max_u_diff,
    })

    # These should be floating-point reconstruction noise only.
    if max_prob_diff > 5e-5 or max_u_diff > 5e-5:
        raise RuntimeError(
            f"Fold {fold}: checkpoint/model reconstruction does not match the official "
            f"OOF outputs (max prob diff={max_prob_diff:.6g}, "
            f"max uncertainty diff={max_u_diff:.6g})."
        )

    print(f"Official OOF validation passed: max probability diff={max_prob_diff:.3g}, "
          f"max uncertainty diff={max_u_diff:.3g}")

    all_rows.append(merged)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

mc_results = pd.concat(all_rows, ignore_index=True)
validation_summary = pd.DataFrame(validation_rows)

mc_results["MC_MEAN_PREDICTED_CLASS"] = (
    mc_results["MC_MEAN_P_pMCI"] >= CLASSIFICATION_THRESHOLD
).astype(np.int64)
mc_results["MC_MEAN_ERROR"] = (
    mc_results["MC_MEAN_PREDICTED_CLASS"] != mc_results["TARGET"]
).astype(np.int64)

result_filename = (
    "quick_test_fold0_mc_dropout_participants.csv"
    if QUICK_TEST else "five_fold_mc_dropout_participants.csv"
)
mc_results.to_csv(TABLE_DIR / result_filename, index=False)
validation_summary.to_csv(TABLE_DIR / "checkpoint_reconstruction_validation.csv", index=False)

display(validation_summary)
display(mc_results.head())
print("\nSaved:", TABLE_DIR / result_filename)


In [ ]:
# 7. Compare uncertainty methods on the SAME deterministic errors
# Raw numerical scales are not compared. Ranking/utility is compared through
# error-detection AUC/AP, correct-vs-incorrect separation, and risk-coverage.

UNCERTAINTY_METHODS = {
    "Evidential uncertainty": "FINAL_UNCERTAINTY",
    "Conflict-aware evidential": "CONFLICT_AWARE_UNCERTAINTY",
    "MC predictive entropy": "MC_PREDICTIVE_ENTROPY",
    "MC mutual information": "MC_MUTUAL_INFORMATION",
    "MC pMCI probability SD": "MC_P_pMCI_STD",
}

def error_detection_summary(df):
    errors = df["ERROR"].to_numpy(dtype=int)
    if np.unique(errors).size < 2:
        raise ValueError("Error-detection AUC is undefined because only one error class is present.")

    rows = []
    for name, column in UNCERTAINTY_METHODS.items():
        scores = df[column].to_numpy(dtype=float)
        correct = scores[errors == 0]
        incorrect = scores[errors == 1]
        rows.append({
            "METHOD": name,
            "COLUMN": column,
            "N": len(df),
            "ERRORS": int(errors.sum()),
            "ERROR_PREVALENCE": float(errors.mean()),
            "ERROR_DETECTION_ROC_AUC": float(roc_auc_score(errors, scores)),
            "ERROR_DETECTION_AVERAGE_PRECISION": float(average_precision_score(errors, scores)),
            "MEAN_CORRECT": float(correct.mean()),
            "MEAN_INCORRECT": float(incorrect.mean()),
            "INCORRECT_MINUS_CORRECT": float(incorrect.mean() - correct.mean()),
        })
    return pd.DataFrame(rows)

error_detection = error_detection_summary(mc_results)
display(error_detection.round(6))

name = "quick_test_uncertainty_comparison.csv" if QUICK_TEST else "uncertainty_comparison.csv"
error_detection.to_csv(TABLE_DIR / name, index=False)

if not QUICK_TEST:
    evid_auc = float(error_detection.loc[
        error_detection["METHOD"] == "Evidential uncertainty",
        "ERROR_DETECTION_ROC_AUC",
    ].iloc[0])
    conflict_auc = float(error_detection.loc[
        error_detection["METHOD"] == "Conflict-aware evidential",
        "ERROR_DETECTION_ROC_AUC",
    ].iloc[0])

    print("\nThesis-result reproduction check:")
    print(f"  Evidential:      {evid_auc:.4f} (reported 0.6572)")
    print(f"  Conflict-aware:  {conflict_auc:.4f} (reported 0.7214)")


In [ ]:
# 8. Deterministic prediction vs MC-averaged prediction
# This is secondary. The primary UQ comparison above uses the same deterministic
# errors for every score. Here we simply check whether MC averaging changes the
# classifier's discrimination/calibration.

def probability_metrics(df, probability_column):
    y = df["TARGET"].to_numpy(dtype=int)
    p = np.clip(df[probability_column].to_numpy(dtype=float), 1e-10, 1 - 1e-10)
    pred = (p >= CLASSIFICATION_THRESHOLD).astype(int)
    return {
        "PROBABILITY": probability_column,
        "ROC_AUC": float(roc_auc_score(y, p)),
        "AVERAGE_PRECISION": float(average_precision_score(y, p)),
        "BRIER": float(brier_score_loss(y, p)),
        "NEGATIVE_LOG_LIKELIHOOD": float(log_loss(y, np.column_stack([1-p, p]), labels=[0, 1])),
        "ERROR_RATE": float(np.mean(pred != y)),
    }

prediction_quality = pd.DataFrame([
    probability_metrics(mc_results, "FINAL_P_pMCI"),
    probability_metrics(mc_results, "MC_MEAN_P_pMCI"),
])
display(prediction_quality.round(6))
prediction_quality.to_csv(TABLE_DIR / "prediction_quality_deterministic_vs_mc_mean.csv", index=False)


In [ ]:
# 9. Paired participant-level bootstrap: MC score vs existing scores

def paired_bootstrap_difference(df, score_a, score_b, metric_name):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    y = df["ERROR"].to_numpy(dtype=int)
    a = df[score_a].to_numpy(dtype=float)
    b = df[score_b].to_numpy(dtype=float)
    n = len(df)

    if metric_name == "ROC_AUC":
        metric = roc_auc_score
    elif metric_name == "AVERAGE_PRECISION":
        metric = average_precision_score
    else:
        raise ValueError(metric_name)

    observed = float(metric(y, a) - metric(y, b))
    diffs = []
    for _ in range(BOOTSTRAP_ITERATIONS):
        idx = rng.integers(0, n, size=n)
        yb = y[idx]
        if np.unique(yb).size < 2:
            continue
        diffs.append(float(metric(yb, a[idx]) - metric(yb, b[idx])))

    diffs = np.asarray(diffs, dtype=float)
    return {
        "SCORE_A": score_a,
        "SCORE_B": score_b,
        "METRIC": metric_name,
        "A_MINUS_B": observed,
        "CI_2.5": float(np.percentile(diffs, 2.5)),
        "CI_97.5": float(np.percentile(diffs, 97.5)),
        "BOOTSTRAP_SAMPLES": len(diffs),
    }

comparisons = []
for mc_col in ["MC_PREDICTIVE_ENTROPY", "MC_MUTUAL_INFORMATION", "MC_P_pMCI_STD"]:
    for existing_col in ["FINAL_UNCERTAINTY", "CONFLICT_AWARE_UNCERTAINTY"]:
        for metric in ["ROC_AUC", "AVERAGE_PRECISION"]:
            comparisons.append(
                paired_bootstrap_difference(mc_results, mc_col, existing_col, metric)
            )

bootstrap_table = pd.DataFrame(comparisons)
display(bootstrap_table.round(6))
bootstrap_table.to_csv(TABLE_DIR / "paired_bootstrap_mc_vs_existing_uncertainty.csv", index=False)


In [ ]:
# 10. Risk-coverage, uncertainty quintiles, and modality-count association

def risk_coverage_table(df, score_column):
    ordered = df.sort_values(score_column, ascending=True).reset_index(drop=True)
    rows = []
    for coverage in np.arange(1.00, 0.49, -0.05):
        n_keep = max(1, int(np.floor(len(ordered) * coverage)))
        kept = ordered.iloc[:n_keep]
        y = kept["TARGET"].to_numpy(dtype=int)
        p = kept["FINAL_P_pMCI"].to_numpy(dtype=float)
        rows.append({
            "SCORE": score_column,
            "COVERAGE": float(coverage),
            "N_RETAINED": len(kept),
            "ERROR_RATE": float(kept["ERROR"].mean()),
            "RETAINED_ROC_AUC": float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan,
        })
    return pd.DataFrame(rows)

risk_coverage = pd.concat(
    [risk_coverage_table(mc_results, col) for col in UNCERTAINTY_METHODS.values()],
    ignore_index=True,
)
risk_coverage.to_csv(TABLE_DIR / "risk_coverage_all_uncertainty_methods.csv", index=False)

quintile_rows = []
for method, col in UNCERTAINTY_METHODS.items():
    temp = mc_results[["ERROR", col]].copy()
    temp["QUINTILE"] = pd.qcut(
        temp[col].rank(method="first"), q=5, labels=False
    ) + 1
    for q, g in temp.groupby("QUINTILE"):
        quintile_rows.append({
            "METHOD": method,
            "SCORE": col,
            "QUINTILE": int(q),
            "N": len(g),
            "MEAN_SCORE": float(g[col].mean()),
            "ERROR_RATE": float(g["ERROR"].mean()),
        })
quintiles = pd.DataFrame(quintile_rows)
quintiles.to_csv(TABLE_DIR / "error_rate_by_uncertainty_quintile.csv", index=False)

availability_rows = []
for method, col in UNCERTAINTY_METHODS.items():
    rho, p = spearmanr(mc_results["ORIGINAL_MODALITY_COUNT"], mc_results[col])
    availability_rows.append({
        "METHOD": method,
        "SCORE": col,
        "SPEARMAN_RHO_MODALITY_COUNT": float(rho),
        "P_VALUE": float(p),
    })
availability = pd.DataFrame(availability_rows)
availability.to_csv(TABLE_DIR / "uncertainty_vs_modality_count.csv", index=False)

score_columns = list(UNCERTAINTY_METHODS.values())
spearman_matrix = mc_results[score_columns].corr(method="spearman")
spearman_matrix.to_csv(TABLE_DIR / "uncertainty_score_spearman_correlations.csv")

display(availability.round(6))
display(spearman_matrix.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
for method, col in UNCERTAINTY_METHODS.items():
    temp = risk_coverage.loc[risk_coverage["SCORE"] == col]
    ax.plot(temp["COVERAGE"], temp["ERROR_RATE"], marker="o", label=method)
ax.set_xlabel("Coverage (fraction retained)")
ax.set_ylabel("Error rate among retained participants")
ax.set_title("Risk-coverage comparison")
ax.invert_xaxis()
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "risk_coverage_mc_vs_existing_uncertainty.png", dpi=180)
plt.show()


## Optional branch-level attribution

Disabled by default because it multiplies inference cost. It uses the same dropout seeds for the full and ablated runs to reduce Monte Carlo noise in branch deltas.


In [ ]:
# 11. OPTIONAL: leave-one-branch-out MC-dropout attribution
# This is a perturbation/sensitivity attribution, not a causal ranking.

def mc_metrics_with_fixed_seeds(model, batch, branch_masks, seeds):
    enable_only_neural_dropout(model)
    samples = []
    with torch.inference_mode():
        for seed in seeds:
            torch.manual_seed(int(seed))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed))
            out = model(
                modalities=batch["modalities"],
                original_branch_masks=branch_masks,
            )
            samples.append(out["final_output"]["probabilities"].detach().float().cpu().numpy())

    probs = np.stack(samples, axis=0)
    mean_p = probs.mean(axis=0)
    pe = entropy(mean_p)
    ee = entropy(probs).mean(axis=0)
    mi = np.maximum(pe - ee, 0.0)
    sd = probs[:, :, 1].std(axis=0, ddof=1 if len(seeds) > 1 else 0)
    return mean_p[:, 1], pe, mi, sd

if RUN_BRANCH_ATTRIBUTION:
    attribution_rows = []

    for fold in OUTER_FOLDS:
        _, loader = make_test_loader(fold)
        model, _ = load_fixed_model(fold)

        for batch_index, batch_cpu in enumerate(loader):
            batch = move_nested_batch_to_device(batch_cpu, DEVICE)
            base_masks = batch["branch_masks"].float()
            rid = batch["rid"].detach().cpu().numpy().reshape(-1)

            seeds = [
                7000000 + fold * 100000 + batch_index * 1000 + i
                for i in range(ATTRIBUTION_MC_SAMPLES)
            ]
            full_p, full_pe, full_mi, full_sd = mc_metrics_with_fixed_seeds(
                model, batch, base_masks, seeds
            )

            base_masks_np = base_masks.detach().cpu().numpy()
            for modality_index, modality in enumerate(MODALITY_ORDER):
                ablated_masks = base_masks.clone()
                ablated_masks[:, modality_index] = 0.0
                abl_p, abl_pe, abl_mi, abl_sd = mc_metrics_with_fixed_seeds(
                    model, batch, ablated_masks, seeds
                )

                for i in range(len(rid)):
                    if base_masks_np[i, modality_index] <= 0.5:
                        continue
                    attribution_rows.append({
                        "FOLD": fold,
                        "RID": int(rid[i]),
                        "MODALITY": modality,
                        "ABS_DELTA_P_pMCI": float(abs(abl_p[i] - full_p[i])),
                        "DELTA_PREDICTIVE_ENTROPY": float(abl_pe[i] - full_pe[i]),
                        "DELTA_MUTUAL_INFORMATION": float(abl_mi[i] - full_mi[i]),
                        "DELTA_P_pMCI_STD": float(abl_sd[i] - full_sd[i]),
                    })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    attribution_participant = pd.DataFrame(attribution_rows)
    attribution_summary = (
        attribution_participant.groupby("MODALITY", as_index=False)
        .agg(
            N=("RID", "count"),
            MEAN_ABS_DELTA_P=("ABS_DELTA_P_pMCI", "mean"),
            MEDIAN_ABS_DELTA_P=("ABS_DELTA_P_pMCI", "median"),
            MEAN_DELTA_PREDICTIVE_ENTROPY=("DELTA_PREDICTIVE_ENTROPY", "mean"),
            MEAN_DELTA_MUTUAL_INFORMATION=("DELTA_MUTUAL_INFORMATION", "mean"),
            MEAN_DELTA_P_STD=("DELTA_P_pMCI_STD", "mean"),
        )
        .sort_values("MEAN_ABS_DELTA_P", ascending=False)
    )

    attribution_participant.to_csv(TABLE_DIR / "mc_dropout_branch_attribution_participant_level.csv", index=False)
    attribution_summary.to_csv(TABLE_DIR / "mc_dropout_branch_attribution_summary.csv", index=False)
    display(attribution_summary.round(6))
else:
    print("Branch attribution is OFF. Set RUN_BRANCH_ATTRIBUTION=True only after the main MC analysis works.")


## How to read the results

The primary table compares every uncertainty score against the **same deterministic fixed-fusion errors**. This is the clean comparison with the dissertation analysis.

- `FINAL_UNCERTAINTY`: the original evidential uncertainty, based on Dirichlet strength.
- `CONFLICT_AWARE_UNCERTAINTY`: the exact thesis score combining final uncertainty with maximum pairwise modality conflict.
- `MC_PREDICTIVE_ENTROPY`: total uncertainty of the MC-averaged class distribution.
- `MC_MUTUAL_INFORMATION`: disagreement across dropout samples; this is the most useful MC-dropout proxy for model/epistemic uncertainty.
- `MC_P_pMCI_STD`: instability of the pMCI probability across stochastic forward passes.

Do **not** compare their raw magnitudes. Compare error-detection ROC AUC/AP, correct-vs-incorrect separation, risk-coverage, and participant ranking/correlation.

The checkpoint-reconstruction cell is deliberately strict. Before any MC result is trusted, it verifies that the deterministic model rebuilt in Reproduces the already saved official out-of-fold probability and evidential uncertainty to floating-point tolerance.
